# Setup

In [1]:

import sys
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

# Make scripts/ importable (notebook lives in notebooks/, scripts/ is its sibling)
sys.path.append(str(Path.cwd().parent))
from scripts.evaluation import evaluate

RANDOM_STATE = 25
DATA_DIR    = Path.cwd().parent / 'data' / 'processed'
RESULTS_DIR = Path.cwd().parent / 'outputs' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load and separate X / y

In [2]:
df = pd.read_parquet(DATA_DIR / 'accidents_clean.parquet')

y = df['Victims_Condition']
X = df.drop(columns=['Victims_Condition'])

print(f"Total:    {len(df):,} rows, {X.shape[1]} features")
print(f"\nTarget distribution:")
print(y.value_counts(normalize=True).round(4))

Total:    463,083 rows, 21 features

Target distribution:
Victims_Condition
With injured victims    0.7173
Without victims         0.2150
With dead victims       0.0677
Name: proportion, dtype: float64


# Stratified 80/20 split + save

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train.to_parquet(DATA_DIR / 'X_train.parquet', index=False)
X_test .to_parquet(DATA_DIR / 'X_test.parquet',  index=False)
y_train.to_frame().to_parquet(DATA_DIR / 'y_train.parquet', index=False)
y_test .to_frame().to_parquet(DATA_DIR / 'y_test.parquet',  index=False)

print(f"Train: {len(X_train):,} rows")
print(f"Test:  {len(X_test):,} rows")
print(f"\nTrain class ratio:\n{y_train.value_counts(normalize=True).round(4)}")
print(f"\nTest class ratio:\n{y_test.value_counts(normalize=True).round(4)}")

Train: 370,466 rows
Test:  92,617 rows

Train class ratio:
Victims_Condition
With injured victims    0.7173
Without victims         0.2150
With dead victims       0.0677
Name: proportion, dtype: float64

Test class ratio:
Victims_Condition
With injured victims    0.7173
Without victims         0.2150
With dead victims       0.0677
Name: proportion, dtype: float64


# Dummy classifier baseline

In [4]:
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(np.zeros((len(y_train), 1)), y_train)

y_pred_dummy = dummy.predict(np.zeros((len(y_test), 1)))

# Evaluate the dummy + save baseline row

In [5]:
results = evaluate(
    y_test,
    y_pred_dummy,
    model_name='DummyClassifier (most_frequent)',
)

results.to_csv(RESULTS_DIR / 'baseline_dummy.csv', index=False)
results


DummyClassifier (most_frequent)
Accuracy : 0.7173
Macro-F1 : 0.2785
MCC      : 0.0000
  F1 [With dead victims]: 0.0000
  F1 [With injured victims]: 0.8354
  F1 [Without victims]: 0.0000

Confusion matrix:
                           pred_With dead victims  pred_With injured victims  \
true_With dead victims                          0                       6269   
true_With injured victims                       0                      66436   
true_Without victims                            0                      19912   

                           pred_Without victims  
true_With dead victims                        0  
true_With injured victims                     0  
true_Without victims                          0  


,model,accuracy,macro_f1,mcc,f1_With dead victims,f1_With injured victims,f1_Without victims
0,DummyClassifier (most_frequent),0.71732,0.278465,0.0,0.0,0.835394,0.0
